In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, lower, regexp_extract, year, desc, count
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.window import Window
import pyspark.sql.functions as F

In [9]:
spark = SparkSession.builder.appName("PopularProgrammingLanguages").getOrCreate()
spark

In [10]:
# Загружаем список языков
languages_df = spark.read.csv("programming-languages.csv", header=True)
languages_df.show(5)
# Приводим к нижнему регистру
languages_list = [row['name'].lower() for row in languages_df.collect()]

+----------+--------------------+
|      name|       wikipedia_url|
+----------+--------------------+
|   A# .NET|https://en.wikipe...|
|A# (Axiom)|https://en.wikipe...|
|A-0 System|https://en.wikipe...|
|        A+|https://en.wikipe...|
|       A++|https://en.wikipe...|
+----------+--------------------+
only showing top 5 rows


In [11]:
# Читаем посты
posts = spark.read.format("xml").option("rowTag", "row").load("posts_sample.xml")
posts.show(5)

+-----------------+------------+--------------------+-----------+-------------+--------------------+--------------------+--------------+---+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+--------------------+--------------------+----------+
|_AcceptedAnswerId|_AnswerCount|               _Body|_ClosedDate|_CommentCount| _CommunityOwnedDate|       _CreationDate|_FavoriteCount|_Id|   _LastActivityDate|       _LastEditDate|_LastEditorDisplayName|_LastEditorUserId|_OwnerDisplayName|_OwnerUserId|_ParentId|_PostTypeId|_Score|               _Tags|              _Title|_ViewCount|
+-----------------+------------+--------------------+-----------+-------------+--------------------+--------------------+--------------+---+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+--------------------+--------------------+-

In [14]:
# Обработка данных
posts_processed = posts.select(
    F.year(F.col("_CreationDate")).alias("year"),
    F.col("_Tags").alias("tags")
).filter("tags is not null and year >= 2010 and year <= 2020")
posts_processed.show(5, truncate=False)

+----+--------------------------------------+
|year|tags                                  |
+----+--------------------------------------+
|2010|<c++><character-encoding>             |
|2010|<sharepoint><infopath>                |
|2010|<iphone><app-store><in-app-purchase>  |
|2010|<symfony1><schema><doctrine><fixtures>|
|2010|<java>                                |
+----+--------------------------------------+
only showing top 5 rows


In [16]:
# Для поиска пересечения тегов поста со списком языков
def extract_langs(tags):
    if not tags: return []
    # Убираем скобки и делим
    cleaned_tags = tags.strip('<>').split('><')
    return [t for t in cleaned_tags if t.lower() in languages_list]

extract_langs_udf = F.udf(extract_langs, "array<string>")

# Разворачиваем теги в строки
top_langs = posts_processed \
    .withColumn("lang", F.explode(extract_langs_udf(F.col("tags")))) \
    .groupBy("year", "lang") \
    .count() \
    .orderBy("year", F.desc("count"))

top_langs.show(5, truncate=False)

+----+-----------+-----+
|year|lang       |count|
+----+-----------+-----+
|2010|java       |52   |
|2010|php        |46   |
|2010|javascript |44   |
|2010|python     |26   |
|2010|objective-c|23   |
+----+-----------+-----+
only showing top 5 rows


In [19]:
window_spec = Window.partitionBy("year").orderBy(F.desc("count"))

final_report = top_langs.withColumn("rank", F.row_number().over(window_spec)) \
    .filter("rank <= 10") \
    .select("year", "lang", "count")

# Сохраняем в Parquet
final_report.write.mode("overwrite").parquet("top_languages_report.parquet")

final_report.show(100)

+----+-----------+-----+
|year|       lang|count|
+----+-----------+-----+
|2010|       java|   52|
|2010|        php|   46|
|2010| javascript|   44|
|2010|     python|   26|
|2010|objective-c|   23|
|2010|          c|   20|
|2010|       ruby|   12|
|2010|     delphi|    8|
|2010|applescript|    3|
|2010|          r|    3|
|2011|        php|  102|
|2011|       java|   93|
|2011| javascript|   83|
|2011|     python|   37|
|2011|objective-c|   34|
|2011|          c|   24|
|2011|       ruby|   20|
|2011|       perl|    9|
|2011|     delphi|    8|
|2011|       bash|    7|
|2012|        php|  154|
|2012| javascript|  132|
|2012|       java|  124|
|2012|     python|   69|
|2012|objective-c|   45|
|2012|       ruby|   27|
|2012|          c|   27|
|2012|       bash|   10|
|2012|          r|    9|
|2012|      scala|    6|
|2013|        php|  198|
|2013| javascript|  198|
|2013|       java|  194|
|2013|     python|   90|
|2013|objective-c|   40|
|2013|          c|   36|
|2013|       ruby|   32|
